In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from dateutil import parser as dtparser
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('sample.csv')

In [ ]:
df.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,amt
0,TXN-3E3A44E74819,USR0020,7457.251963207814,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31,NaN
1,TXN-84B8FC890498,USR0043,308.51678134343047,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249,NaN
2,TXN-95B71538FFAD,USR0044,₹3159,1710796175,jaipur,Jaipur,NaN,D??,web,Card,168910.49,success,192.73.41.151,NaN
3,TXN-657F2702B5B2,USR0060,5349.73,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119,NaN
4,TXN-84C8BBF69256,USR0020,2454.5652128035144,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   transaction_id         1447 non-null   object 
 1   user_id                1447 non-null   object 
 2   transaction_amount     1374 non-null   object 
 3   transaction_timestamp  1447 non-null   object 
 4   user_location          1447 non-null   object 
 5   merchant_location      1447 non-null   object 
 6   merchant_category      1369 non-null   object 
 7   device_id              1374 non-null   object 
 8   device_type            1447 non-null   object 
 9   payment_method         1373 non-null   object 
 10  account_balance        1373 non-null   float64
 11  transaction_status     1447 non-null   object 
 12  ip_address             1368 non-null   object 
 13  amt                    49 non-null     object 
dtypes: float64(1), object(13)
memory usage: 158.4+ KB


In [ ]:
df.dtypes

,0
transaction_id,object
user_id,object
transaction_amount,object
transaction_timestamp,object
user_location,object
merchant_location,object
merchant_category,object
device_id,object
device_type,object
payment_method,object


In [ ]:
print('Number of Rows :-',df.shape[0])
print('Number of Columns :-',df.shape[1])

Number of Rows :- 1447
Number of Columns :- 14


In [ ]:
#Get Overall Statistics About The DataFrame
df.describe()

,account_balance
count,1373.000000
mean,73386.952403
std,55553.341123
min,0.000000
25%,23885.610000
50%,68661.030000
75%,121395.610000
max,191245.660000


Separating Numerical and Categorical Columns

In [ ]:
num_col=df.select_dtypes(include=('int','float')).columns
num_col

Index(['account_balance'], dtype='object')

In [ ]:
cat_col=df.select_dtypes(include=('object')).columns
cat_col

Index(['transaction_id', 'user_id', 'transaction_amount',
       'transaction_timestamp', 'user_location', 'merchant_location',
       'merchant_category', 'device_id', 'device_type', 'payment_method',
       'transaction_status', 'ip_address', 'amt'],
      dtype='object')

Handling Missing Values

In [ ]:
df.isnull().sum()/len(df)*100

,0
transaction_id,0.000000
user_id,0.000000
transaction_amount,5.044921
transaction_timestamp,0.000000
user_location,0.000000
merchant_location,0.000000
merchant_category,5.390463
device_id,5.044921
device_type,0.000000
payment_method,5.114029


Standard Rules:-
If column contains more than 40% null value, then remove these column.

If column contains 4 to 40% null value, then we will replace those column with mean or median

If column contains less than 4 to 5% , then we will remove the rows with null values

In [ ]:
def clean_amount(v):
    if pd.isna(v): return None
    v = str(v).strip()
    v = re.sub(r'[₹RsINR,\s]', '', v, flags=re.IGNORECASE)
    try:    return float(v)
    except: return None

# Merge shadow column FIRST, then parse both together
df['transaction_amount'] = df['transaction_amount'].fillna(df['amt'])
df.drop(columns=['amt'], inplace=True)
df['transaction_amount'] = df['transaction_amount'].apply(clean_amount)

# Verify
print(df['transaction_amount'].dtype)          # → float64
print(df['transaction_amount'].isna().sum())   # → should be ~73
print(df['transaction_amount'].describe())

float64
24
count    1.423000e+03
mean     1.616143e+04
std      1.400065e+05
min      0.000000e+00
25%      1.532291e+03
50%      3.259990e+03
75%      6.041254e+03
max      1.678107e+06
Name: transaction_amount, dtype: float64


In [ ]:
def parse_ts(v):
    if pd.isna(v): return pd.NaT
    v = str(v).strip()

    # Unix epoch — 10 digit integer string e.g. "1717321977"
    if re.match(r'^\d{10}$', v):
        return pd.to_datetime(int(v), unit='s')

    # Compact format — 14 digit e.g. "20240421030940"
    if re.match(r'^\d{14}$', v):
        return pd.to_datetime(v, format='%Y%m%d%H%M%S')

    # DD/MM/YYYY HH:MM:SS — pandas gets this wrong without explicit format
    if re.match(r'^\d{2}/\d{2}/\d{4}', v):
        return pd.to_datetime(v, format='%d/%m/%Y %H:%M:%S', errors='coerce')

    # MM-DD-YYYY HH:MM  e.g. "02-24-2024 10:48"
    if re.match(r'^\d{2}-\d{2}-\d{4}', v):
        return pd.to_datetime(v, format='%m-%d-%Y %H:%M', errors='coerce')

    # DD-Mon-YYYY  e.g. "04-Feb-2024"
    if re.match(r'^\d{2}-[A-Za-z]{3}-\d{4}', v):
        return pd.to_datetime(v, format='%d-%b-%Y', errors='coerce')

    # All other natural language + ISO8601 → let dateutil handle
    try:    return pd.to_datetime(dtparser.parse(v))
    except: return pd.NaT

df['transaction_timestamp'] = df['transaction_timestamp'].apply(parse_ts)

# Verify
print(df['transaction_timestamp'].dtype)          # → datetime64[ns]
print("Unparseable:", df['transaction_timestamp'].isna().sum())  # → ideally 0

datetime64[ns]
Unparseable: 0


In [ ]:
df['account_balance'] = df['account_balance'].fillna(df['account_balance'].median())
print(df['account_balance'].isna().sum())  # → 0

0


In [ ]:
# Canonical category list
CATEGORY_MAP = {
    'travel':        'Travel',
    'tr':            'Travel',
    't':             'Travel',
    'education':     'Education',
    'edu':           'Education',
    'utilities':     'Utilities',
    'utili':         'Utilities',
    'fuel':          'Fuel',
    'fu':            'Fuel',
    'electronics':   'Electronics',
    'clothing':      'Clothing',
    'cl':            'Clothing',
    'clothin':       'Clothing',
    'grocery':       'Grocery',
    'groce':         'Grocery',
    'gr':            'Grocery',
    'food & dining': 'Food & Dining',
    'food & di':     'Food & Dining',
    'food & d':      'Food & Dining',
    'food':          'Food & Dining',
    'entertainment': 'Entertainment',
    'enterta':       'Entertainment',
    'ent':           'Entertainment',
    'healthcare':    'Healthcare',
    'healthca':      'Healthcare',
    'h':             'Healthcare',
}

def clean_category(v):
    if pd.isna(v): return 'Unknown'
    # Strip noise chars, lowercase, strip
    v = re.sub(r'[#\.\?]+$', '', str(v)).strip().lower()
    # Exact match first
    if v in CATEGORY_MAP:
        return CATEGORY_MAP[v]
    # Prefix match — longest prefix wins
    best = None
    for key in CATEGORY_MAP:
        if v.startswith(key):
            if best is None or len(key) > len(best):
                best = key
    return CATEGORY_MAP[best] if best else v.title()

df['merchant_category'] = df['merchant_category'].apply(clean_category)
df['merchant_category'] = df['merchant_category'].astype('category')

# Verify
print(df['merchant_category'].value_counts())

merchant_category
Electronics      181
Travel           149
Utilities        148
Clothing         140
Grocery          132
Entertainment    130
Fuel             129
Healthcare       122
Education        120
Food & Dining    116
Unknown           78
C                  1
Ut                 1
Name: count, dtype: int64


In [ ]:
print("=== FINAL DTYPES ===")
print(df.dtypes)

print("\n=== REMAINING NULLS ===")
print(df.isnull().sum())

print("\n=== SHAPE ===")
print(df.shape)

=== FINAL DTYPES ===
transaction_id                   object
user_id                          object
transaction_amount              float64
transaction_timestamp    datetime64[ns]
user_location                    object
merchant_location                object
merchant_category              category
device_id                        object
device_type                      object
payment_method                   object
account_balance                 float64
transaction_status               object
ip_address                       object
dtype: object

=== REMAINING NULLS ===
transaction_id            0
user_id                   0
transaction_amount       24
transaction_timestamp     0
user_location             0
merchant_location         0
merchant_category         0
device_id                73
device_type               0
payment_method           74
account_balance           0
transaction_status        0
ip_address               79
dtype: int64

=== SHAPE ===
(1447, 13)


In [ ]:
print("\n=== REMAINING NULLS ===")
df.isnull().sum()/len(df)*100


=== REMAINING NULLS ===


,0
transaction_id,0.000000
user_id,0.000000
transaction_amount,1.658604
transaction_timestamp,0.000000
user_location,0.000000
merchant_location,0.000000
merchant_category,0.000000
device_id,5.044921
device_type,0.000000
payment_method,5.114029


In [ ]:
print("\n=== SHAPE ===")
print(df.shape)


=== SHAPE ===
(1447, 13)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   transaction_id         1447 non-null   object        
 1   user_id                1447 non-null   object        
 2   transaction_amount     1423 non-null   float64       
 3   transaction_timestamp  1447 non-null   datetime64[ns]
 4   user_location          1447 non-null   object        
 5   merchant_location      1447 non-null   object        
 6   merchant_category      1447 non-null   category      
 7   device_id              1374 non-null   object        
 8   device_type            1447 non-null   object        
 9   payment_method         1373 non-null   object        
 10  account_balance        1447 non-null   float64       
 11  transaction_status     1447 non-null   object        
 12  ip_address             1368 non-null   object        
dtypes: 

In [ ]:
df

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,jaipur,Jaipur,Unknown,D??,web,Card,168910.49,success,192.73.41.151
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-02-07 03:50:23.000000,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,TXN-09C97C1A8F6A,USR0069,2672.714350,2024-02-26 16:50:59.387741,LKO,lucknow,Electronics,DEV-8657F3E2,web,Wallet,118582.16,success,192.38.168.146
1443,TXN-C83C897D358E,USR0043,615.000000,2024-05-14 16:57:04.000000,Pune,Ahmedabad,Clothing,ATO-7D32C693,web,Card,119773.72,success,140.18.71.75
1444,TXN-A442B398E85E,USR0022,956.784542,2024-02-16 05:47:47.000000,Mumbai,ahmedabad,Entertainment,DEV-54C67087,mobile,Card,73970.71,success,192.62.225.88
1445,TXN-C0877E9A01B8,USR0048,2332.363564,2024-02-07 01:02:50.000000,Chennai,HYD,Healthcare,DEV-D7256DA2,ATM,UPI,11946.20,pending,192.109.213.177


In [ ]:
df.columns

Index(['transaction_id', 'user_id', 'transaction_amount',
       'transaction_timestamp', 'user_location', 'merchant_location',
       'merchant_category', 'device_id', 'device_type', 'payment_method',
       'account_balance', 'transaction_status', 'ip_address'],
      dtype='object')

In [ ]:
df.isnull().sum() / len(df) * 100

,0
transaction_id,0.000000
user_id,0.000000
transaction_amount,1.658604
transaction_timestamp,0.000000
user_location,0.000000
merchant_location,0.000000
merchant_category,0.000000
device_id,5.044921
device_type,0.000000
payment_method,5.114029


In [ ]:
df['device_id'] = df['device_id'].fillna('unknown')
df['payment_method'] = df['payment_method'].fillna('unknown')
df['ip_address'] = df['ip_address'].fillna('unknown')

In [ ]:
def clean_amount(x):
    if pd.isna(x):
        return 0
    x = str(x)
    x = re.sub(r'[^\d.]', '', x)
    return float(x) if x != '' else 0

df['transaction_amount'] = df['transaction_amount'].apply(clean_amount)

In [ ]:
df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'], errors='coerce')

df['hour'] = df['transaction_timestamp'].dt.hour
df['day'] = df['transaction_timestamp'].dt.dayofweek

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['is_fraud'] = df['transaction_status'].apply(
    lambda x: 1 if str(x).lower() == 'failed' else 0
)

Encoding categorical columns

In [ ]:
from sklearn.preprocessing import LabelEncoder

cols = [
    'user_location', 'merchant_location', 'merchant_category',
    'device_id', 'device_type', 'payment_method', 'ip_address'
]

le = LabelEncoder()
for col in cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
df.isnull().sum()/len(df)*100

,0
transaction_id,0.0
user_id,0.0
transaction_amount,0.0
transaction_timestamp,0.0
user_location,0.0
merchant_location,0.0
merchant_category,0.0
device_id,0.0
device_type,0.0
payment_method,0.0


In [ ]:
df.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,hour,day,is_fraud
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,31,34,9,169,1,0,140227.79,success,1232,11,3,0
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,20,13,9,198,1,2,122548.77,success,236,9,6,0
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,62,21,10,27,2,0,168910.49,success,1078,21,0,0
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-02-07 03:50:23.000000,2,2,2,121,2,2,9046.00,success,750,3,2,0
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,62,21,12,169,1,0,153095.35,success,1223,23,2,0


In [ ]:
df['is_fraud'].value_counts()

,count
is_fraud,
0,1315
1,111


In [ ]:
import os

os.makedirs('/content/drive/MyDrive/Aivengers/data', exist_ok=True)

In [ ]:
df.to_csv('/content/drive/MyDrive/Aivengers/data/cleaned.csv', index=False)